In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os
captions_file = '/content/drive/MyDrive/ROCO/data/train/radiology/captions.txt'
images_path = '/content/drive/MyDrive/ROCO/data/train/radiology/images'

data = []
with open(captions_file, 'r') as f:
    for line in f:
        line = line.strip()
        if not line or '\t' not in line:
            print(f"Skipping line: {line}")
            continue
        try:
            image_id, caption = line.split('\t')
            data.append([image_id, caption, images_path])
        except ValueError:
            print(f"Error processing line: {line}")
df = pd.DataFrame(data, columns=['Image ID', 'Image Caption', 'Image Path'])
print(df.head(5))

Skipping line: ROCO_42292
     Image ID                                      Image Caption  \
0  ROCO_00002   Computed tomography scan in axial view showin...   
1  ROCO_00003   Bacterial contamination occurred after comple...   
2  ROCO_00004   The patient had residual paralysis of the han...   
3  ROCO_00005      Panoramic radiograph after immediate loading.   
4  ROCO_00007   Plain abdomen x-ray: Multiple air levels at t...   

                                          Image Path  
0  /content/drive/MyDrive/ROCO/data/train/radiolo...  
1  /content/drive/MyDrive/ROCO/data/train/radiolo...  
2  /content/drive/MyDrive/ROCO/data/train/radiolo...  
3  /content/drive/MyDrive/ROCO/data/train/radiolo...  
4  /content/drive/MyDrive/ROCO/data/train/radiolo...  


In [ ]:
def classify_caption(caption):
    abnormal_keywords = [
        'tumor', 'cancer', 'fracture', 'lesion', 'abnormal', 'pathology', 'disease', 'infection', 'swelling', 'blood', 'bleeding', 'mass', 'clot', 'nodules', 'metastasis', 'sarcoma', 'stroke', 'pneumonia', 'hemorrhage', 'infarction', 'diabetes', 'cyst', 'aneurysm', 'calcification', 'ischemia', 'tuberculosis', 'fibrosis', 'sclerosis', 'laceration', 'dislocation', 'nodular', 'congestion', 'edema', 'pneumothorax', 'collapse', 'deformity',
        'scarring', 'stent', 'abscess', 'bronchitis', 'pneumonitis', 'retinopathy', 'osteopenia', 'osteoporosis', 'cardiomegaly', 'calcified', 'pericardial', 'hepatomegaly', 'splenomegaly', 'lymphadenopathy', 'nephropathy', 'colitis', 'gastropathy', 'sepsis', 'chronic', 'necrosis', 'myocardial', 'pulmonary embolism', 'hyperplasia', 'atrophy', 'osteomyelitis', 'autoimmune', 'malignancy', 'neoplasm', 'hemangioma', 'pleural effusion', 'sarcoidosis', 'granuloma',
        'peritonitis', 'hypertension', 'hemodialysis', 'fibrotic', 'lymphoma', 'meningioma', 'diabetic retinopathy', 'cholecystitis', 'pleuritis', 'glaucoma', 'subdural hematoma', 'subarachnoid hemorrhage', 'pneumonectomy', 'retinal detachment', 'myelopathy', 'bronchiectasis', 'lymphocytic', 'reflux', 'pericarditis', 'gallstones', 'splenic infarction', 'biliary obstruction', 'aplastic anemia', 'acute', 'cholecystectomy', 'lymphopenia', 'hemophilia', 'sickle cell', 'asthma',
        'allergy', 'bronchospasm', 'corticosteroids', 'antibiotics', 'hemoptysis', 'peptic ulcer', 'herpes', 'hiv', 'aids', 'candidiasis', 'hyperthyroidism', 'hypothyroidism', 'osteosarcoma', 'epilepsy', 'seizure', 'pneumonitis', 'cardiomyopathy', 'metastatic', 'adenocarcinoma', 'radiation', 'chemotherapy', 'malaria', 'chronic fatigue syndrome', 'bipolar disorder', 'depression', 'schizophrenia', 'anemia', 'hepatitis', 'cirrhosis', 'gout', 'eczema', 'psoriasis', 'lupus', 'pseudomonas',
        'legionella', 'septicemia', 'hyperventilation', 'tachycardia', 'bradycardia', 'dysrhythmia', 'carcinoma', 'hemochromatosis', 'diabetic ketoacidosis', 'hydrocephalus', 'cystic fibrosis', 'multiple sclerosis', 'polyneuropathy', 'encephalopathy', 'autoimmune disease', 'gastroparesis', 'endometriosis', 'colorectal cancer', 'irritable bowel syndrome', 'amyloidosis', 'psoriatic arthritis', 'interstitial lung disease', 'parkinson\'s', 'meningitis', 'encephalitis', 'pneumoniae', 'tracheitis',
        'pleural effusion', 'pericardial effusion', 'mesothelioma', 'peripheral neuropathy', 'hypovolemia', 'hyperkalemia', 'hypokalemia', 'hypernatremia', 'hyponatremia', 'hemorrhagic shock', 'cholestasis', 'malabsorption', 'dehydration', 'intussusception', 'intubation', 'hypoxia', 'hypocapnia', 'severe headache', 'nausea', 'vomiting', 'hemodynamic instability', 'cerebrovascular accident', 'hysterectomy', 'ovarian cysts', 'pancreatitis', 'deep vein thrombosis', 'pulmonary fibrosis', 'stent graft',
        'cardiac arrest', 'chronic obstructive pulmonary disease', 'diabetic retinopathy', 'gastropathy', 'blood clot', 'hemophilia', 'endocarditis', 'cirrhosis', 'rheumatoid arthritis', 'hepatomegaly', 'splenomegaly', 'abdominal pain', 'melena', 'epigastric pain', 'bronchiolitis', 'hypoalbuminemia', 'hydropneumothorax', 'gastrectomy', 'vulvovaginal candidiasis', 'pleural plaque', 'sick sinus syndrome', 'thrombophlebitis', 'pneumoconiosis', 'emphysema', 'benign prostatic hyperplasia', 'hyperlipidemia', 'gallbladder', 'cystic adenoma', 'neuroblastoma', 'bladder cancer', 'renal failure', 'ascending aortic aneurysm', 'sinusitis', 'menorrhagia', 'gastroesophageal reflux disease', 'adrenal insufficiency', 'fibrosarcoma', 'vesicoureteral reflux', 'lumbosacral strain'
    ]
    for keyword in abnormal_keywords:
        if keyword.lower() in caption.lower():
            return 1
    return 0
df['label'] = df['Image Caption'].apply(classify_caption)

NORMAL --> 0           
ABNORMAL --> 1

In [ ]:
random_sample = df.sample(n=5)
print(random_sample)

         Image ID                                      Image Caption  \
22290  ROCO_27779   Contrast enhanced MRI of the left knee demons...   
52007  ROCO_64966   After 65 days from the afatinib 40 mg adminis...   
33090  ROCO_41334   Demonstrates bony infill with the nasopharyng...   
3139   ROCO_03907   Angiography, sagittal view, showing the absen...   
18850  ROCO_23514   The CT scan obtained on the fourteenth hospit...   

                                              Image Path  label  
22290  /content/drive/MyDrive/ROCO/data/train/radiolo...      1  
52007  /content/drive/MyDrive/ROCO/data/train/radiolo...      1  
33090  /content/drive/MyDrive/ROCO/data/train/radiolo...      0  
3139   /content/drive/MyDrive/ROCO/data/train/radiolo...      0  
18850  /content/drive/MyDrive/ROCO/data/train/radiolo...      0  


In [ ]:
train_texts = df['Image Caption'][:8000]
test_texts = df['Image Caption'][8000:10000]

train_labels = df['label'][:8000]
test_labels = df['label'][8000:10000]

In [ ]:
train_texts.head()

,Image Caption
0,Computed tomography scan in axial view showin...
1,Bacterial contamination occurred after comple...
2,The patient had residual paralysis of the han...
3,Panoramic radiograph after immediate loading.
4,Plain abdomen x-ray: Multiple air levels at t...


In [ ]:
test_labels.head()

,label
8000,1
8001,1
8002,0
8003,1
8004,0


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts(train_texts)

In [ ]:
train_sequences=tokenizer.texts_to_sequences(train_texts)
test_sequences=tokenizer.texts_to_sequences(test_texts)

In [ ]:
train_sequences

[[21, 18, 14, 5, 25, 24, 7, 1278, 2, 1, 8, 262, 123],
 [4527,
  6341,
  2200,
  26,
  1629,
  2,
  350,
  265,
  201,
  5,
  1,
  497,
  110,
  2022,
  6,
  4,
  4528,
  263,
  83,
  617,
  295],
 [1,
  33,
  321,
  430,
  4529,
  2,
  1,
  726,
  26,
  6342,
  231,
  27,
  4530,
  12,
  4531,
  1,
  1630,
  6,
  1032,
  12,
  1,
  967,
  1443,
  76,
  27,
  6343,
  45,
  3657,
  4,
  338,
  36,
  1,
  64,
  6344,
  98,
  1,
  218,
  3,
  274,
  2684,
  1,
  1744,
  16,
  1,
  181,
  1079,
  2,
  1,
  338,
  242,
  39,
  545],
 [151, 23, 26, 911, 2685],
 [178,
  58,
  30,
  31,
  87,
  137,
  765,
  17,
  1,
  328,
  58,
  37,
  63,
  968,
  596,
  3,
  63,
  137,
  766,
  1,
  669],
 [4,
  65,
  39,
  38,
  878,
  6,
  2686,
  6345,
  25,
  570,
  13,
  347,
  4,
  2403,
  2404,
  32,
  259,
  12,
  1,
  322,
  440,
  227,
  1,
  554,
  3102,
  37,
  6,
  466,
  19,
  670,
  1171,
  6,
  554,
  2687,
  570,
  822,
  15,
  149,
  269,
  5,
  1,
  8,
  6346,
  36,
  1223,
  2023,
  503]

In [ ]:
word_index=tokenizer.word_index

In [ ]:
max_sequence_length=max(len(seq) for seq in train_sequences)
max_sequence_length

205

In [ ]:
train_padded=pad_sequences(train_sequences,maxlen=max_sequence_length,padding="post")
test_padded=pad_sequences(test_sequences,maxlen=max_sequence_length,padding="post")

In [ ]:
train_padded

array([[  21,   18,   14, ...,    0,    0,    0],
       [4527, 6341, 2200, ...,    0,    0,    0],
       [   1,   33,  321, ...,    0,    0,    0],
       ...,
       [ 354,  230,  418, ...,    0,    0,    0],
       [   4,   43,   24, ...,    0,    0,    0],
       [1334,   39,   38, ...,    0,    0,    0]], dtype=int32)

In [ ]:
train_labels=to_categorical(train_labels)
test_labels=to_categorical(test_labels)

In [ ]:
train_labels

array([[1., 0.],
       [1., 0.],
       [1., 0.],
       ...,
       [0., 1.],
       [1., 0.],
       [0., 1.]])

In [ ]:
import numpy as np
from gensim.models import Word2Vec
w2v_model=Word2Vec(sentences=[text.split() for text in train_texts],vector_size=100)

In [ ]:
embedding_dim=100
embedding_matrix=np.zeros((len(word_index)+1,embedding_dim))

In [ ]:
for word,i in word_index.items():
  if word in w2v_model.wv:
    embedding_matrix[i]=w2v_model.wv[word]

In [ ]:
embedding_matrix

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.56134892,  0.41877285,  0.03274717, ..., -0.51011091,
         0.03819739,  0.45008832],
       [-0.25035095,  0.39853352,  0.02397735, ..., -0.30027306,
        -0.23297808,  0.21549325],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]])

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense

In [ ]:
model=Sequential([
        Embedding(input_dim=len(word_index)+1,
                  output_dim=embedding_dim,
                  weights=[embedding_matrix],
        trainable=False),
        LSTM(128),
        Dense(2,activation="softmax")
])

In [ ]:
model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

In [ ]:
model.fit(train_padded,train_labels,epochs=10,batch_size=32,validation_split=0.2)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 65s 323ms/step - accuracy: 0.5613 - loss: 0.6859 - val_accuracy: 0.5663 - val_loss: 0.6849
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 64s 320ms/step - accuracy: 0.5481 - loss: 0.6886 - val_accuracy: 0.5663 - val_loss: 0.6844
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 82s 319ms/step - accuracy: 0.5576 - loss: 0.6865 - val_accuracy: 0.5663 - val_loss: 0.6848
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 64s 318ms/step - accuracy: 0.5598 - loss: 0.6859 - val_accuracy: 0.5663 - val_loss: 0.6848
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 82s 317ms/step - accuracy: 0.5626 - loss: 0.6854 - val_accuracy: 0.5663 - val_loss: 0.6845
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 84s 326ms/step - accuracy: 0.5552 - loss: 0.6868 - val_accuracy: 0.5663 - val_loss: 0.6844
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 68s 340ms/step - accuracy: 0.5561 - loss: 0.6870 - val_accuracy: 0.5663 - val_loss: 0.6848
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 84s 352ms/step - accuracy: 0.5571 - loss: 0

In [ ]:
test_loss,test_accuracy=model.evaluate(test_padded,test_labels)

63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 98ms/step - accuracy: 0.5457 - loss: 0.6891


In [ ]:
test_accuracy

0.5490000247955322

In [ ]:
import random
import numpy as np
random_caption = train_texts.iloc[random.randint(0, len(train_texts)-1)]

random_sequence = tokenizer.texts_to_sequences([random_caption])
random_padded = pad_sequences(random_sequence, maxlen=max_sequence_length, padding='post')

prediction = model.predict(random_padded)

predicted_class = np.argmax(prediction)

print("Random Caption:", random_caption)
print("Predicted Class:", "Abnormal" if predicted_class == 0 else "Normal")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
Random Caption:  chest X-ray of case 4 showing right middle and lower lobe opacity
Predicted Class: Abnormal


GRU

In [ ]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense


In [ ]:
df = df.sample(n=10000, random_state=42)

In [ ]:
tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(df['Image Caption'])

In [ ]:
sequences = tokenizer.texts_to_sequences(df['Image Caption'])

In [ ]:
max_length = max(len(seq) for seq in sequences)

padded_sequences = pad_sequences(sequences, maxlen=max_length, padding="post")

In [ ]:
num_classes = 2
categorical_labels = to_categorical(df['label'], num_classes=num_classes)

In [ ]:
embedding_dim = 64

model = Sequential([

    Embedding(input_dim=1000, output_dim=embedding_dim),
    GRU(128, dropout=0.2),
    Dense(num_classes, activation="softmax")
])

In [ ]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=['accuracy'])

In [ ]:
model.fit(padded_sequences, categorical_labels, epochs=10)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 101s 314ms/step - accuracy: 0.5621 - loss: 0.6854
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 314ms/step - accuracy: 0.5652 - loss: 0.6852
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 138s 302ms/step - accuracy: 0.5704 - loss: 0.6839
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 93s 297ms/step - accuracy: 0.5637 - loss: 0.6853
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 143s 299ms/step - accuracy: 0.5700 - loss: 0.6840
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 148s 317ms/step - accuracy: 0.5661 - loss: 0.6850
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 137s 300ms/step - accuracy: 0.5656 - loss: 0.6852
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 300ms/step - accuracy: 0.5680 - loss: 0.6841
Epoch 9/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 94s 299ms/step - accuracy: 0.5735 - loss: 0.6825
Epoch 10/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 299ms/step - accuracy: 0.5723 - loss: 0.6830


In [ ]:
accuracy = model.evaluate(padded_sequences, categorical_labels, verbose=1)[1]
print(f"Accuracy: {accuracy * 100:.2f}%")

313/313 ━━━━━━━━━━━━━━━━━━━━ 21s 66ms/step - accuracy: 0.5673 - loss: 0.6840
Accuracy: 56.85%


In [ ]:
captions = df['Image Caption'].head(144).tolist()
data = '\n'.join([caption.strip() + '\\n' for caption in captions])

In [ ]:
print(data)

Computed tomography scan in axial view showing obliteration of the left maxillary sinus\n
Bacterial contamination occurred after completion of root canal treatment in the tooth, which remained with a temporary filling for 15 month.\n
The patient had residual paralysis of the hand after poliomyelitis. It was necessary to stabilize the thumb with reference to the index finger. This was accomplished by placing a graft from the bone bank between the first and second metacarpals. The roentgenogram shows the complete healing of the graft one year later.\n
Panoramic radiograph after immediate loading.\n
Plain abdomen x-ray: Multiple air levels at the mid-abdomen (arrows), no radiopaque shadow, and no air under the diaphragm.\n
A 3-year-old child with visual difficulties. Axial FLAIR image show a supra-sellar lesion extending to the temporal lobes along the optic tracts (arrows) with moderate mass effect, compatible with optic glioma. FLAIR hyperintensity is also noted in the left mesencephalo

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer=Tokenizer()
tokenizer.fit_on_texts([data])

In [ ]:
vocab_size=len(tokenizer.word_index)+1
vocab_size

1147

In [ ]:
sequences=[]

In [ ]:
for line in data.split("\n"):
  token_list=tokenizer.texts_to_sequences([line])[0]
  for i in range(1,len(token_list)):
    ngram_sequence=token_list[:i+1]
    sequences.append(ngram_sequence)

In [ ]:
sequences

[[36, 37],
 [36, 37, 19],
 [36, 37, 19, 4],
 [36, 37, 19, 4, 22],
 [36, 37, 19, 4, 22, 24],
 [36, 37, 19, 4, 22, 24, 8],
 [36, 37, 19, 4, 22, 24, 8, 398],
 [36, 37, 19, 4, 22, 24, 8, 398, 3],
 [36, 37, 19, 4, 22, 24, 8, 398, 3, 1],
 [36, 37, 19, 4, 22, 24, 8, 398, 3, 1, 15],
 [36, 37, 19, 4, 22, 24, 8, 398, 3, 1, 15, 126],
 [36, 37, 19, 4, 22, 24, 8, 398, 3, 1, 15, 126, 57],
 [36, 37, 19, 4, 22, 24, 8, 398, 3, 1, 15, 126, 57, 2],
 [399, 400],
 [399, 400, 401],
 [399, 400, 401, 18],
 [399, 400, 401, 18, 199],
 [399, 400, 401, 18, 199, 3],
 [399, 400, 401, 18, 199, 3, 86],
 [399, 400, 401, 18, 199, 3, 86, 66],
 [399, 400, 401, 18, 199, 3, 86, 66, 48],
 [399, 400, 401, 18, 199, 3, 86, 66, 48, 4],
 [399, 400, 401, 18, 199, 3, 86, 66, 48, 4, 1],
 [399, 400, 401, 18, 199, 3, 86, 66, 48, 4, 1, 402],
 [399, 400, 401, 18, 199, 3, 86, 66, 48, 4, 1, 402, 87],
 [399, 400, 401, 18, 199, 3, 86, 66, 48, 4, 1, 402, 87, 403],
 [399, 400, 401, 18, 199, 3, 86, 66, 48, 4, 1, 402, 87, 403, 6],
 [399, 400, 

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_sequence_len=max([len(seq) for seq in sequences])
sequences=pad_sequences(sequences,maxlen=max_sequence_len,padding="pre")
sequences

array([[  0,   0,   0, ...,   0,  36,  37],
       [  0,   0,   0, ...,  36,  37,  19],
       [  0,   0,   0, ...,  37,  19,   4],
       ...,
       [  0,   0,   0, ..., 221,   3,   9],
       [  0,   0,   0, ...,   3,   9,  71],
       [  0,   0,   0, ...,   9,  71,   2]], dtype=int32)

In [ ]:
from tensorflow.keras.utils import to_categorical
X=sequences[:,:-1]
y=sequences[:,-1]
y=to_categorical(y,num_classes=vocab_size)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense

model=Sequential([
    Embedding(vocab_size,50),
    LSTM(100,return_sequences=False),
    Dense(vocab_size,activation="softmax")

])

In [ ]:
model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=['accuracy'])

In [ ]:
model.fit(X,y,epochs=100)

Epoch 1/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 95ms/step - accuracy: 0.1437 - loss: 4.8944
Epoch 2/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 11s 107ms/step - accuracy: 0.1585 - loss: 4.7487
Epoch 3/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 81ms/step - accuracy: 0.1575 - loss: 4.6835
Epoch 4/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 82ms/step - accuracy: 0.1641 - loss: 4.5438
Epoch 5/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 13s 110ms/step - accuracy: 0.1754 - loss: 4.4580
Epoch 6/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 91ms/step - accuracy: 0.1856 - loss: 4.4157
Epoch 7/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 91ms/step - accuracy: 0.1842 - loss: 4.2921
Epoch 8/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 12s 107ms/step - accuracy: 0.2037 - loss: 4.1697
Epoch 9/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 83ms/step - accuracy: 0.2026 - loss: 4.1580
Epoch 10/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 10s 106ms/step - accuracy: 0.2178 - loss: 4.0710
Epoch 11/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 9s 96ms/step - accuracy: 0.2196 - loss: 3.9595
Epoch 12/100
92/92 ━━━━━━━━━━━━━━━━━━━━ 8s 

In [ ]:
import numpy as np
def generate_text(seed_text,next_words,max_sequence_len):
  for _ in range(next_words):
    token_list=tokenizer.texts_to_sequences([seed_text])[0]
    token_list=pad_sequences([token_list],maxlen=max_sequence_len-1,padding="pre")
    predicted=np.argmax(model.predict(token_list,verbose=0),axis=-1)
    output_word=""
    for word,index in tokenizer.word_index.items():
      if index==predicted:
        output_word=word
        break
    seed_text+=" "+output_word
  return seed_text

In [ ]:
Actual_caption = "Panoramic radiograph after immediate loading."
seed_text="Panoramic radiograph after"
generate_text(seed_text,next_words=2,max_sequence_len=max_sequence_len)

'Panoramic radiograph after mild mild'

In [ ]:
Actual_caption = "Conventional design for hybrid prosthesis with long distal cantilevers."
seed_text="Conventional design for hybrid prosthesis with"
generate_text(seed_text,next_words=3,max_sequence_len=max_sequence_len)

'Conventional design for hybrid prosthesis with long distal cantilevers'

In [ ]:
Actual_Caption = "A computed tomography scan showing a mass in the gallbladder."
seed_text="A computed tomography scan showing "
generate_text(seed_text,next_words=5,max_sequence_len=max_sequence_len)